# Лекция: Факторный анализ в Python

**Дисциплина:** Введение в анализ больших данных

**Факторный анализ** снижает размерность: много наблюдаемых переменных объясняются меньшим числом **латентных факторов**.

- **нагрузки (loadings)** — связь переменных с факторами;
- **общность (communality)** — доля дисперсии, объяснённая факторами;
- **собственные значения** — вклад факторов;
- **вращение (varimax)** — для интерпретируемости.

Инструменты: `sklearn.decomposition.FactorAnalysis`, собственные значения / scree, varimax.

Демо: синтетический опрос о качестве сервиса (не из лабораторного задания).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(7)
print("Библиотеки загружены")


---
## 1. Данные

Девять пунктов шкалы 1–5. Заложены три латентных фактора:  
«качество», «доставка», «поддержка».


In [ ]:
n = 120
Q = np.random.normal(0, 1, n)
D = np.random.normal(0, 1, n)
S = np.random.normal(0, 1, n)

items = pd.DataFrame({
    "product_quality": 0.80*Q + 0.10*D + np.random.normal(0, 0.35, n),
    "packaging":       0.70*Q + 0.15*D + np.random.normal(0, 0.40, n),
    "value_for_money": 0.65*Q + 0.20*S + np.random.normal(0, 0.45, n),
    "delivery_speed":  0.15*Q + 0.80*D + np.random.normal(0, 0.35, n),
    "tracking":        0.10*Q + 0.75*D + np.random.normal(0, 0.40, n),
    "pack_condition":  0.20*Q + 0.70*D + np.random.normal(0, 0.40, n),
    "response_time":   0.10*D + 0.80*S + np.random.normal(0, 0.35, n),
    "staff_politeness":0.15*Q + 0.75*S + np.random.normal(0, 0.40, n),
    "issue_resolve":   0.85*S + np.random.normal(0, 0.40, n),
})
items = (3.2 + items).clip(1, 5)
print(items.describe().round(2).T[["mean", "std", "min", "max"]])


### Корреляционная матрица


In [ ]:
corr = items.corr()
plt.figure(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True)
plt.title("Корреляции пунктов")
plt.tight_layout()
plt.show()


---
## 2. Число факторов: scree и parallel analysis

Критерий Кайзера: собственные значения > 1.  
Parallel analysis: сравниваем с собственными значениями случайных данных.


In [ ]:
eigvals = np.linalg.eigvalsh(corr.values)[::-1]
print("Собственные значения:", np.round(eigvals, 3))

n_perm = 40
rand_eigs = []
for _ in range(n_perm):
    R = StandardScaler().fit_transform(np.random.normal(size=items.shape))
    C = np.corrcoef(R.T)
    rand_eigs.append(np.linalg.eigvalsh(C)[::-1])
rand_eigs = np.mean(rand_eigs, axis=0)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(eigvals)+1), eigvals, "o-", label="данные")
plt.plot(range(1, len(rand_eigs)+1), rand_eigs, "s--", label="случайные")
plt.axhline(1, color="gray", ls=":", label="Kaiser (=1)")
plt.xlabel("номер фактора")
plt.ylabel("собственное значение")
plt.title("Scree + parallel analysis")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("По parallel analysis (eig > random):", int(np.sum(eigvals > rand_eigs)))


---
## 3. Оценка факторной модели

`FactorAnalysis` + вращение **varimax**.


In [ ]:
def varimax(Phi, gamma=1.0, q=30, tol=1e-6):
    p, k = Phi.shape
    R = np.eye(k)
    d = 0.0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma / p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = u @ vh
        d = float(np.sum(s))
        if d_old and d / d_old < 1 + tol:
            break
    return Phi @ R

def fit_fa(data, n_factors):
    fa = FactorAnalysis(n_components=n_factors, random_state=0, max_iter=2000)
    fa.fit(data)
    loadings = varimax(fa.components_.T)
    df = pd.DataFrame(
        loadings, index=data.columns,
        columns=[f"F{i+1}" for i in range(n_factors)],
    )
    df["communality"] = (loadings ** 2).sum(axis=1)
    return df, fa


In [ ]:
print("=== 2 фактора ===")
load2, _ = fit_fa(items, 2)
print(load2.round(3))

print("\n=== 3 фактора ===")
load3, _ = fit_fa(items, 3)
print(load3.round(3))


In [ ]:
plt.figure(figsize=(5, 6))
sns.heatmap(load3.drop(columns=["communality"]), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Нагрузки (3 фактора, varimax)")
plt.tight_layout()
plt.show()


### Как читать нагрузки

- |нагрузка| ≳ 0.4–0.5 → переменная связана с фактором;  
- высокая **communality** → пункт хорошо объясняется факторами;  
- дайте факторам **содержательные** имена по сильным нагрузкам.


---
## 4. KMO (пригодность данных для FA)

Желательно KMO > 0.6.


In [ ]:
def kmo_simple(corr):
    C = np.asarray(corr, float).copy()
    inv = np.linalg.inv(C)
    pcorr = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv)))
    np.fill_diagonal(pcorr, 0)
    np.fill_diagonal(C, 0)
    r2 = (C ** 2).sum()
    p2 = (pcorr ** 2).sum()
    return r2 / (r2 + p2)

print(f"KMO ≈ {kmo_simple(corr.values):.3f}")


### Как описать результаты

1. Корреляции и KMO — уместен ли FA.  
2. Scree / parallel — сколько факторов.  
3. Таблица нагрузок после varimax — интерпретация и имена факторов.  
4. Communality — насколько хорошо каждый пункт объясняется.

---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Корреляции | `df.corr()` |
| Собственные значения | `np.linalg.eigvalsh(corr)[::-1]` |
| FA | `FactorAnalysis(n_components=k).fit(X)` |
| Нагрузки | `fa.components_.T` (+ varimax) |
| Communality | `(loadings**2).sum(axis=1)` |

---
## Что сделать после лекции

1. Повторите scree, 2- и 3-факторные модели на **других** шкалах.  
2. Откройте лабораторное задание и выполните FA **самостоятельно** на своём CSV.  
3. Число факторов выбирайте и по числам, и по интерпретируемости.

Удачи!
